In [2]:
import arcpy
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Function to compute Jaccard between two boolean lists
def jaccard(a, b):
    intersection = sum(x and y for x, y in zip(a, b))
    union = sum(x or y for x, y in zip(a, b))
    return intersection / union if union else 0

In [4]:
def dice(a, b):
    intersection = sum(x and y for x, y in zip(a, b))
    count_a = sum(a)
    count_b = sum(b)
    denom = count_a + count_b
    return (2 * intersection / denom) if denom else 0

In [5]:
# --- SETTINGS ---
proj = arcpy.mp.ArcGISProject("CURRENT")
mp = proj.listMaps()[0]

In [6]:
import os
os.getcwd()

'C:\\Users\\Mina\\Desktop\\Mina Files\\PostDoc Vienna\\Codes\\vague_cognitive_regions\\Sahara\\Sahara'

In [7]:
layer = mp.listLayers("Sahara_Grid20_All")[0]   # name shown in TOC
print(layer)

fc = layer.dataSource   # gets full path automatically
print(fc)

Sahara_Grid20_All
C:\Users\Mina\Desktop\Mina Files\PostDoc Vienna\Codes\vague_cognitive_regions\Sahara\Data\Sahara_Grid20_All\Sahara_Grid20_All


In [8]:
fields = ["B_GPT4o_En", "B_DS_En"]

# Read all relevant fields into memory
data = {f: [] for f in fields}

with arcpy.da.SearchCursor(fc, fields) as cursor:
    for row in cursor:
        for i, f in enumerate(fields):
            data[f].append(row[i] == "TRUE")   # convert to True/False

In [9]:
# Compute full Jaccard matrix
jaccard_matrix = []
for f1 in fields:
    row = []
    for f2 in fields:
        row.append(jaccard(data[f1], data[f2]))
    jaccard_matrix.append(row)

# Convert to Pandas DataFrame for clean printing
df_jaccard = pd.DataFrame(jaccard_matrix, columns=fields, index=fields)

print("\nFull Jaccard Similarity Matrix:\n")
print(df_jaccard.round(2))  # print with 2 decimals


Full Jaccard Similarity Matrix:

            B_GPT4o_En  B_DS_En
B_GPT4o_En        1.00     0.23
B_DS_En           0.23     1.00


In [10]:
df_jaccard.to_csv(r"C:\Users\Mina\Desktop\Mina Files\PostDoc Vienna\Codes\vague_cognitive_regions\Sahara\Sahara\Sahara_Grids_JaccardIndices.csv", index=False)

In [11]:
dice_matrix = []
for f1 in fields:
    row = []
    for f2 in fields:
        row.append(dice(data[f1], data[f2]))
    dice_matrix.append(row)

df_dice = pd.DataFrame(dice_matrix, columns=fields, index=fields)

print("\nFull Dice Similarity Matrix:\n")
print(df_dice.round(2))


Full Dice Similarity Matrix:

            B_GPT4o_En  B_DS_En
B_GPT4o_En        1.00     0.37
B_DS_En           0.37     1.00


In [12]:
df_dice.to_csv(r"C:\Users\Mina\Desktop\Mina Files\PostDoc Vienna\Codes\vague_cognitive_regions\Sahara\Sahara\Sahara_Grids_DiceCoefficients.csv", index=False)

In [13]:
numeric_fields = ["S_GPT4o_En", "S_DS_En"]

# Read all numeric values into a dict
data = {f: [] for f in numeric_fields}

with arcpy.da.SearchCursor(fc, numeric_fields) as cursor:
    for row in cursor:
        for i, f in enumerate(numeric_fields):
            data[f].append(row[i])

# Convert to DataFrame for convenience
df_pearson = pd.DataFrame(data)

In [14]:
# Use pandas built-in method
pearson_matrix = df_pearson.corr(method='pearson')

print("Full Pearson Correlation Matrix:\n")
print(pearson_matrix.round(2))

Full Pearson Correlation Matrix:

            S_GPT4o_En  S_DS_En
S_GPT4o_En         1.0      0.6
S_DS_En            0.6      1.0


In [15]:
df_pearson.to_csv(r"C:\Users\Mina\Desktop\Mina Files\PostDoc Vienna\Codes\vague_cognitive_regions\Sahara\Sahara\Sahara_Grids_PearsonCorrelations.csv", index=False)